In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')

(Tutorial_Convert)=
# Convert

*Converting a molecular system into another form or multiple forms.*

In MolSysMT, a molecular system can be represented in many different forms (files, third-party objects, or native data structures). The function {func}`molsysmt.basic.convert` provides a form-agnostic way to convert a molecular system from any supported input form into any target form.

:::{hint}
Visit the section {ref}`Introduction_Forms` if you are not familiar with the concept of "form" in MolSysMT.
:::

:::{versionadded} 1.0.0
:::

## How this function works

```{admonition} API documentation
Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.basic.convert`.
```

Let's explore how conversion works starting with single-item conversions, moving to composite multi-item inputs, and concluding with splitting systems into multiple output forms.

In [2]:
import molsysmt as msm

## Basic Conversion: Converting a single item

### Single-step form conversion

Let's start with a single molecular system from the local dataset (`181l.bcif.gz`) and convert it across several forms:

In [3]:
molsys_A = msm.systems['T4 lysozyme L99A']['181l.bcif.gz']
molsys_B = msm.convert(molsys_A, to_form='molsysmt.MolSys')
molsys_C = msm.convert(molsys_B, to_form='string:pdb_text')
molsys_D = msm.convert(molsys_B, to_form='mdtraj.Trajectory')
molsys_E = msm.convert(molsys_D, to_form='openmm.Topology')

:::{tip}
All methods defined in the {ref}`molsysmt.basic <API basic>` module can also be invoked from the library's top level. Hence, {func}`molsysmt.convert` is the same method as {func}`molsysmt.basic.convert`.
:::

:::{tip}
The default target form of {func}`molsysmt.basic.convert` when `to_form` is omitted is the native `molsysmt.MolSys`.
:::

We can inspect the form of any converted system using {func}`molsysmt.basic.get_form` or `msm.info()`:

In [4]:
msm.info(molsys_B)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_waters,n_ions,n_small_molecules,n_proteins,n_structures
molsysmt.MolSys,1441,302,141,6,141,5,136,2,2,1,1


In [5]:
msm.get_form(molsys_D)

'mdtraj.Trajectory'

### Converting a selected subset

Conversion can also be restricted to a specific atom selection. Here we convert only the benzene ligand of T4 lysozyme into a PDB text string:

In [6]:
molsys_sel = msm.convert(molsys_B, to_form='string:pdb_text', selection='molecule_name=="BENZENE"')

In [7]:
print(molsys_sel)

HEADER    MOLECULAR SYSTEM                        06-AUG-26       
REMARK   1 Created by MolSysMT version 1.0 on 06-AUG-2026 at 07:03:58
CRYST1   60.900   60.900   97.000  90.00  90.00 120.00
ATOM      1 C1   BNZ A 400      25.978   5.327   4.779  0.00 20.05           C  
ATOM      2 C2   BNZ A 400      26.395   5.074   3.499  0.00 21.25           C  
ATOM      3 C3   BNZ A 400      27.340   5.860   2.902  0.00 26.80           C  
ATOM      4 C4   BNZ A 400      27.837   6.921   3.569  0.00 21.02           C  
ATOM      5 C5   BNZ A 400      27.420   7.196   4.856  0.00 25.56           C  
ATOM      6 C6   BNZ A 400      26.498   6.379   5.469  0.00 26.48           C  
CONECT    1    2
CONECT    1    6
CONECT    2    3
CONECT    3    4
CONECT    4    5
CONECT    5    6
END



## Composite Conversions: Combining multiple items into one

Often a molecular system is split across multiple complementary files or objects—for instance, a topology file paired with coordinate or trajectory files. `msm.convert` accepts a list of items and merges their information into a single system.

### Topology and single-structure coordinates

Consider an Amber topology (`.prmtop`) and its corresponding single-structure coordinate file (`.inpcrd`):

In [8]:
prmtop_file = msm.systems['pentalanine']['pentalanine.prmtop']
inpcrd_file = msm.systems['pentalanine']['pentalanine.inpcrd']

Let's inspect the two input items individually:

In [9]:
msm.info(prmtop_file)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_waters,n_peptides,n_structures
file:prmtop,5207,1722,1716,1,1716,2,1715,1,None


In [10]:
msm.info(inpcrd_file)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_structures
file:inpcrd,5207,None,None,None,None,None,1


By passing both items as a list to `msm.convert`, we combine them into a single `molsysmt.MolSys`:

In [11]:
molsys_composite = msm.convert([prmtop_file, inpcrd_file], to_form='molsysmt.MolSys')

In [12]:
msm.info(molsys_composite)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_waters,n_peptides,n_structures
molsysmt.MolSys,5207,1722,1716,1,1716,2,1715,1,1


### Topology and trajectory: Structure axis ownership

When a topology file (which may hold a single reference structure) is combined with a multi-structure trajectory (such as a DCD or XTC file), the trajectory dictates the resulting structure axis.

Let's see this in action using a PSF topology and a DCD trajectory from the `POPC membrane` dataset:

In [13]:
psf_file = msm.systems['POPC membrane']['popc_membrane.psf']
dcd_file = msm.systems['POPC membrane']['popc_membrane.dcd']

When converting `[psf_file, dcd_file]`, notice that the resulting system contains **5 structures** (matching the DCD trajectory):

In [14]:
molsys_traj1 = msm.convert([psf_file, dcd_file], to_form='molsysmt.MolSys')

In [15]:
msm.info(molsys_traj1)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_waters,n_small_molecules,n_lipids,n_structures
molsysmt.MolSys,78974,13532,13532,3,13239,70,13170,68,294,5


Crucially, the order in which items are listed is transparent. Swapping the order to `[dcd_file, psf_file]` produces the exact same system:

In [16]:
molsys_traj2 = msm.convert([dcd_file, psf_file], to_form='molsysmt.MolSys')
msm.compare(molsys_traj1, molsys_traj2, coordinates=True)

True

:::{tip}
Combining a topology item with a trajectory item retrieves the full trajectory structure axis. If your goal is instead to join multiple trajectory files sequentially, use {ref}`Tutorial_Concatenate_structures`.
:::

## Splitting a system into multiple items

You can also split a single input system into multiple target forms simultaneously by passing a list to `to_form`:

In [17]:
molsys_h5 = msm.systems['pentalanine']['traj_pentalanine.h5']
topology, structures = msm.convert(molsys_h5, to_form=['molsysmt.Topology', 'molsysmt.Structures'])

In [18]:
msm.info(topology)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.Topology,62,7,1,1,1,1,1,None


In [19]:
msm.info(structures)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_structures
molsysmt.Structures,62,None,None,None,None,None,5000


Let's visualize the composite system formed by `[topology, structures]` interactively:

In [20]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

molsysviewer_htmlfile = '_static/views/tools_basic_convert.html'

In [21]:
msm.view([topology, structures], structure_indices=3500, standard=True)

'<iframe src="../../../../../../../../../_static/views/tools_basic_convert.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

## Inspecting supported conversions

To query available conversion paths, use {func}`molsysmt.supported.conversions`:

In [22]:
msm.supported.conversions(from_form='mdtraj.Trajectory', to_form_type='string')

,string:alphafold_id,string:amino_acids_1,string:amino_acids_3,string:pdb_id,string:pdb_text,string:smiles,string:uniprot_id
mdtraj.Trajectory,False,True,True,False,False,False,False


In [23]:
msm.supported.conversions(from_form='mdtraj.Trajectory', to_form_type='file', as_rows='to')

,mdtraj.Trajectory
file:bcif,False
file:bcif.gz,False
file:cif,False
file:cif.gz,False
file:crd,False
file:dcd,False
file:fasta,False
file:gro,False
file:h5,False
file:h5msm,False


:::{seealso}
{ref}`Introduction_Forms`:  
List and explain the different forms of molecular systems.

{ref}`user-foundations-entrance-demo-systems`:     
Access a collection of predefined molecular systems for testing and demonstration purposes.

{ref}`Tutorial_Select`:   
Select atoms or other elements from a molecular system.

{ref}`Tutorial_Info`:     
Display information about a molecular system.

{ref}`Tutorial_View`:     
Visualize a molecular system.

{ref}`Tutorial_Concatenate_structures`:  
Join structural data from several systems into a single system with consecutive structures.
:::